# QE band pdos plotting

#### モジュール

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import ipynbname
NB_NAME = ipynbname.name()

#### データセットの定義

In [ ]:
# 公開用サンプル

dataset_material_a = {
    "title": "MaterialA: band - pdos (example)",
    "band": {
        "file": "./../materials/MaterialA/NC/SOC/bands/bands.out",
    },
    "kpath": {
        "ticks":  [0, 10, 20, 30, 40, 50, 60, 70, 80, 90],
        "labels": ["Γ", "X", "S", "Y", "Γ", "Z", "U", "R", "T", "Z"],
    },
    "pdos": [
        {
            "title": "AtomX",
            "files": [
                {"path": "./../materials/MaterialA/NC/SOC/pdos/AtomX_s05_pdos.dat", "label": "AtomX(s0.5)"},
                {"path": "./../materials/MaterialA/NC/SOC/pdos/AtomX_p05_pdos.dat", "label": "AtomX(p0.5)"},
                {"path": "./../materials/MaterialA/NC/SOC/pdos/AtomX_p15_pdos.dat", "label": "AtomX(p1.5)"},
            ],
        },
        {
            "title": "AtomY",
            "files": [
                {"path": "./../materials/MaterialA/NC/SOC/pdos/AtomY_s05_pdos.dat", "label": "AtomY(s0.5)"},
                {"path": "./../materials/MaterialA/NC/SOC/pdos/AtomY_p05_pdos.dat", "label": "AtomY(p0.5)"},
                {"path": "./../materials/MaterialA/NC/SOC/pdos/AtomY_p15_pdos.dat", "label": "AtomY(p1.5)"},
            ],
        },
    ],
}

#### QEバンドファイル読み込み関数

In [ ]:
# bands.outファイル読み取り
def read_qe_band_file(filename):
    with open(filename, 'r') as f:
        content = f.read()
    header = re.search(r'nbnd=\s*(\d+),\s*nks=\s*(\d+)', content)
    nbnd = int(header.group(1))
    nks  = int(header.group(2))
    numbers = np.fromstring(re.sub(r'&plot[^\n]*/\n', '', content), sep=' ')
    energies = numbers.reshape(nks, 3 + nbnd)[:, 3:]  # k座標3列を除く
    return np.arange(nks), energies  # shape: (nks, nbnd)

#### プロット

In [ ]:
# データセットの指定
dataset = dataset_material_a
title = dataset["title"]

# サブプロットフレームの作成
n_panels = 1 + len(dataset["pdos"])
fig, axes = plt.subplots(1, n_panels, figsize=(2.5 * n_panels, 4), sharey=True)
fig.canvas.header_visible = False

# バンドの描画
ax1 = axes[0]
kpoints, bands = read_qe_band_file(dataset["band"]["file"])
for i in range(bands.shape[1]):
    ax1.plot(kpoints, bands[:, i], color='black', linewidth=0.5)
ax1.set_xlabel("K-point")
ax1.set_ylabel("Energy (eV)")
ax1.set_xticks(dataset["kpath"]["ticks"])
ax1.set_xticklabels(dataset["kpath"]["labels"])
ax1.set_xlim(0, max(kpoints))
ax1.grid(True, linestyle=':')

# 状態密度の描画
for idx, atom in enumerate(dataset["pdos"]):
    ax = axes[idx + 1]
    for f in atom["files"]:
        energies, pdos = np.loadtxt(f["path"], usecols=(0, 1), unpack=True)
        ax.plot(pdos, energies, linewidth=0.8, label=f["label"])
    ax.set_xlabel("Density of States")
    ax.set_xlim(0, None)
    ax.set_title(atom["title"])
    ax.legend(fontsize=10)
    ax.grid(True, linestyle=':')

fig.suptitle(title, y=0.95, fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])

# 画像として保存
save_dir = f"./{NB_NAME}_save"
os.makedirs(save_dir, exist_ok=True)
fig.savefig(f"{save_dir}/{re.sub(r'[^\w\-]+', '_', title)}.png", dpi=300, bbox_inches="tight")